# Target speaker enrollment and segment filtering

Workflow:

1. Manually cut 2-3 single-speaker reference clips (~10s each) of the target speaker.
2. Enroll them into a named `SpeakerProfile`.
3. Diarize a full video with any diarizer backend.
4. Score every diarization turn against the profile (cosine similarity of speaker embeddings).
5. Inspect the similarity histogram, pick a threshold, filter (precision-first), export kept segments.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

ENDSWITH = "notebooks"
NOTEBOOK_DIR = os.getcwd()
if not NOTEBOOK_DIR.endswith(ENDSWITH):
    raise ValueError(f"Not in correct dir, expect end with {ENDSWITH}, but got {NOTEBOOK_DIR} instead")
BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

DATA_DIR = os.path.join(BASE_DIR, ".data")

## 1. Get source audio and cut reference clips

Ingest 2-3 videos of the target speaker, listen, and note timestamp ranges where **only the target speaker** talks (clean, no music/overlap). ~10 seconds per clip is plenty.

In [ ]:
from src.yt_crawler import YtCrawler

crawler = YtCrawler(
    output_dir=os.path.join(DATA_DIR, "yt_crawler", "downloads"),
    work_dir=os.path.join(DATA_DIR, "yt_crawler", "work"),
)

ref_audio_1 = crawler.ingest("https://www.youtube.com/watch?v=REFERENCE_VIDEO_1")
ref_audio_2 = crawler.ingest("https://www.youtube.com/watch?v=REFERENCE_VIDEO_2")

ref_audio_1.notebook_display()

In [ ]:
from src.utils.AudioCutter import AudioCutter

cutter = AudioCutter(output_dir=os.path.join(DATA_DIR, "speaker_profiles", "_raw_cuts"))

# Adjust the ranges after listening; each clip must contain ONLY the target speaker.
clip_1 = cutter.cut(ref_audio_1, "1:05", "1:15", unit="timestamp")
clip_2 = cutter.cut(ref_audio_1, "3:40", "3:50", unit="timestamp")
clip_3 = cutter.cut(ref_audio_2, "0:30", "0:40", unit="timestamp")

for clip in (clip_1, clip_2, clip_3):
    clip.notebook_display()

## 2. Enroll the profile

Clips are copied to `.data/speaker_profiles/<name>/clips/`; the profile is reusable across sessions and machines (sync `.data/speaker_profiles/` if needed).

In [ ]:
from src.diarization import SpeakerVerifier

verifier = SpeakerVerifier(device="auto")

profile = verifier.enroll("khanh_vy", [clip_1, clip_2, clip_3], overwrite=True)
print(profile.name, "->", profile.profile_dir)
print(verifier.list_profiles())

## 3. Diarize a target video

Any diarizer backend works; the verifier only needs the turns.

In [ ]:
from src.diarization import PyannoteDiarizer

target_audio = crawler.ingest("https://www.youtube.com/watch?v=TARGET_VIDEO")

with PyannoteDiarizer(device="auto") as diarizer:
    diarization = diarizer.diarize(target_audio)

print(f"{len(diarization.speakers)} speakers, {len(diarization.turns)} turns")

## 4. Score all turns against the profile

Scoring is done once; thresholding afterwards is free, so tune the threshold on the histogram below. A clear bimodal split (target vs everyone else) is the healthy case.

In [ ]:
with verifier:
    scored = verifier.score(target_audio, diarization, profile)

print(f"Scored {len(scored.segments)} segments")

In [ ]:
import matplotlib.pyplot as plt

similarities = [seg.similarity for seg in scored.segments]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(similarities, bins=50, edgecolor="black")
ax.set_xlabel("Cosine similarity to profile centroid")
ax.set_ylabel("Segment count")
ax.set_title(f"Similarity distribution vs profile {scored.profile_name!r}")
plt.show()
plt.close(fig)

# Per-diarization-speaker mean similarity: sanity check which cluster is the target.
by_speaker: dict[str, list[float]] = {}
for seg in scored.segments:
    by_speaker.setdefault(seg.speaker_id, []).append(seg.similarity)
for speaker_id, values in sorted(by_speaker.items()):
    print(f"{speaker_id}: n={len(values)} mean={sum(values)/len(values):.3f} max={max(values):.3f}")

## 5. Filter and export

Precision-first: pick the threshold from the histogram gap (start high, e.g. 0.6, and lower only if too little audio survives). Overlapping-speaker segments and short segments are dropped by default.

In [ ]:
THRESHOLD = 0.6

kept = SpeakerVerifier.filter(
    scored,
    threshold=THRESHOLD,
    min_duration_s=1.5,
    exclude_overlap=True,
)
total_s = sum(seg.duration_s for seg in kept.segments)
print(f"Kept {len(kept.segments)}/{len(scored.segments)} segments, {total_s:.1f}s total")
for seg in kept.segments[:20]:
    print(f"  {seg.start_s:8.2f}s - {seg.end_s:8.2f}s  sim={seg.similarity:.3f}  ({seg.speaker_id})")

In [ ]:
# Spot-check a few kept segments by ear before exporting everything.
export_cutter = AudioCutter(
    output_dir=os.path.join(DATA_DIR, "target_speaker", profile.name, target_audio.source_id)
)

for seg in kept.segments[:5]:
    preview = export_cutter.cut(target_audio, seg.start_s, seg.end_s)
    preview.notebook_display()

In [ ]:
import json
from dataclasses import asdict

# Export all kept segments as wav cuts plus a JSON manifest.
export_dir = export_cutter.output_dir
exported = [export_cutter.cut(target_audio, seg.start_s, seg.end_s) for seg in kept.segments]

manifest_path = os.path.join(export_dir, "segments.json")
os.makedirs(export_dir, exist_ok=True)
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(asdict(kept), f, ensure_ascii=False, indent=2)

print(f"Exported {len(exported)} clips + manifest to {export_dir}")